In [46]:
# Data import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

df = pd.read_excel("pcos_data_base_ml.xlsx")


## Pipeline oczyszczania i walidacji danych PCOS – lista kroków

### Podstawowa lista

1. **Przegląd danych** – kształt, typy, brakujące wartości, rozkład grup
2. **Identyfikacja kolumn indeksowalnych** – subject_id, group, wiek, BMI
3. **Parametry występujące tylko dla jednej grupy** – PCOS vs Control_no_PCOS
4. **Klasyfikacja parametrów do kategorii** – hormonalne, metaboliczne, lipidowe, zapalne, etc.
5. **Parametry z bardzo mało danymi** – N < 10 (do wykluczenia) oraz 10 <= N < 30 (do osobnej analizy)
6. **Wykrywanie błędów formatowania** – przecinki zamiast kropek, tekstowe wartości null
7. **Parametry bez zmienności** – stałe lub prawie stałe (unikalnych wartości ≤ 2)
8. **Identyfikacja jednostek miary** – mg/dL vs mmol/L (glukoza), ng/mL vs μg/L (D-dimer), nmol/L vs pmol/L (SHBG)
9. **Weryfikacja zakresów wartości (outliers)** – wartości spoza fizjologicznych norm (np. CRP > 100, SHBG > 300)
10. **Spójność między kolumnami mierzącymi ten sam parametr** – np. różne kolumny SHBG, testosteronu, prolaktyny
11. **Korelacje i redundancja między parametrami** – usunięcie zbędnych, wybór najlepszego reprezentanta
12. **Analiza patternu braków danych** – MCAR, MAR, czy MNAR
13. **Definicja minimalnej liczebności próby dla analizy** – N ≥ 30, N ≥ 50, czy N ≥ 100
14. **Strategia imputacji brakujących wartości** – mean/median, KNN, MICE, lub usunięcie
15. **Normalizacja / skalowanie przed klastrowaniem** – Z-score, Min-Max, lub log transformacja (dla skośnych rozkładów)


In [47]:
# Initialize quality dictionary for all columns
parameter_quality = {}

for col in df.columns:
    parameter_quality[col] = {
        # Basic info
        'original_name': col,
        'data_type': str(df[col].dtype),
        'n_total': len(df),
        'n_nonnull': df[col].notna().sum(),
        'n_unique': df[col].nunique(),
        
        # Flags (start as True/empty – will be updated when problems found)
        'useful': True,           # Overall flag for further analysis
        'exclude_reason': [],     # List of reasons to exclude
        
        # Specific issues
        'only_in_group': None,    # 'PCOS', 'Control_no_PCOS', or None
        'too_few_data': False,    # N < 30
        'no_variance': False,     # ≤2 unique values
        'formatting_errors': False, # commas, text nulls
        'unit_unknown': False,    # cannot determine units
        'outliers_detected': False, # extreme values
        
        # Metadata for decision making
        'category': None,         # e.g., 'TSH', 'FT4'
        'recommended_action': None, # 'exclude', 'fix', 'merge', 'keep'
        'notes': []               # Free text notes
    }

print(f"Initialized quality dictionary for {len(parameter_quality)} parameters")

Initialized quality dictionary for 224 parameters


In [48]:
# STEP 1: DATA OVERVIEW

print("=" * 80)
print("STEP 1: DATA OVERVIEW")
print("=" * 80)

# Shape
print(f"Shape: {df.shape}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# Data types
print("\nData types:")
print(df.dtypes.value_counts())

# Group distribution
print("\nGroup distribution:")
print(df['group'].value_counts())

# First rows
print("\nFirst 5 rows:")
print(df.head())

# Missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_summary = missing_summary[missing_summary['Missing'] > 0].sort_values('Missing', ascending=False)

print(f"\nMissing values summary:")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values: {len(missing_summary)}")
print(f"\nTop 20 columns with most missing values:")
print(missing_summary.head(20))

STEP 1: DATA OVERVIEW
Shape: (1331, 224)
Rows: 1331
Columns: 224

Data types:
float64           191
str                30
datetime64[us]      2
int64               1
Name: count, dtype: int64

Group distribution:
group
PCOS               1286
Control_no_PCOS      45
Name: count, dtype: int64

First 5 rows:
   Wiek            group subject_id  17 - OH progesteron (L79) (17-OHPG)  \
0    27  Control_no_PCOS   CTRL_001                                  NaN   
1    24  Control_no_PCOS   CTRL_002                                  NaN   
2    23  Control_no_PCOS   CTRL_003                                  NaN   
3    27  Control_no_PCOS   CTRL_004                                  NaN   
4    23  Control_no_PCOS   CTRL_005                                  NaN   

   17 OH progesteron (L79)  ALAT (ALT)  \
0                      NaN         NaN   
1                      NaN         NaN   
2                      NaN         NaN   
3                      NaN         NaN   
4                      Na

In [49]:
# STEP 2: IDENTIFY INDEXABLE COLUMNS

print("\n" + "=" * 80)
print("STEP 2: IDENTIFY INDEXABLE COLUMNS")
print("=" * 80)

indexable_cols = ['group', 'subject_id', 'Nr KG']

found_indexable = []
for col in indexable_cols:
    if col in df.columns:
        found_indexable.append(col)
        print(f"✓ {col}")
    else:
        print(f"✗ {col} (not found)")

# Mark them in parameter_quality
for col in found_indexable:
    if col in parameter_quality:
        parameter_quality[col]['category'] = 'Index'
        parameter_quality[col]['recommended_action'] = 'keep_as_index'
        parameter_quality[col]['notes'].append('Used as identifier/grouping variable')

print(f"\n✅ Identified {len(found_indexable)} indexable columns")


STEP 2: IDENTIFY INDEXABLE COLUMNS
✓ group
✓ subject_id
✓ Nr KG

✅ Identified 3 indexable columns


In [50]:
# STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP

print("\n" + "=" * 80)
print("STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP")
print("=" * 80)

pcos_df = df[df['group'] == 'PCOS']
control_df = df[df['group'] == 'Control_no_PCOS']

for col in df.columns:    
    pcos_count = pcos_df[col].notna().sum()
    control_count = control_df[col].notna().sum()
    
    if pcos_count > 0 and control_count == 0:
        parameter_quality[col]['only_in_group'] = 'PCOS'
        parameter_quality[col]['notes'].append(f'Has {pcos_count} values, but 0 in Control')
        print(f"⚠️ {col}: ONLY IN PCOS ({pcos_count} values)")
        
    elif control_count > 0 and pcos_count == 0:
        parameter_quality[col]['only_in_group'] = 'Control_no_PCOS'
        parameter_quality[col]['notes'].append(f'Has {control_count} values, but 0 in PCOS')
        print(f"⚠️ {col}: ONLY IN Control_no_PCOS ({control_count} values)")


STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP
⚠️ 17 - OH progesteron (L79) (17-OHPG): ONLY IN PCOS (965 values)
⚠️ 17 OH progesteron (L79): ONLY IN PCOS (212 values)
⚠️ ALAT (ALT): ONLY IN PCOS (263 values)
⚠️ AMH (hormon anty-Mullerowski) (AMH_CP): ONLY IN PCOS (976 values)
⚠️ AMH ng/ml: ONLY IN Control_no_PCOS (45 values)
⚠️ AMH-anty Mullerian Hormon (AMH): ONLY IN PCOS (147 values)
⚠️ APTT Czas kaolinowo-kefalinowy (APTTCZ): ONLY IN PCOS (5 values)
⚠️ ASO - ilościowo (ASOIL): ONLY IN PCOS (1 values)
⚠️ ASPAT (AST): ONLY IN PCOS (263 values)
⚠️ Androstendion (ANDRO): ONLY IN PCOS (682 values)
⚠️ Androstendion (I31): ONLY IN PCOS (189 values)
⚠️ Androstendion (I31) (ANDRO): ONLY IN PCOS (255 values)
⚠️ Anty - HCV (L_ANTHCV): ONLY IN PCOS (1 values)
⚠️ Anty-TG (O18): ONLY IN PCOS (21 values)
⚠️ Anty-TG (p/c przeciw tyreoglobulinie) (ATG): ONLY IN PCOS (254 values)
⚠️ Anty-TPO (ATA_TPO): ONLY IN PCOS (1043 values)
⚠️ BMI kg/m2: ONLY IN Control_no_PCOS (45 values)
⚠️ Białko C-reaktywne 

In [51]:
# STEP 4: CLASSIFY PARAMETERS INTO CLINICAL CATEGORIES

import re

print("=" * 80)
print("STEP 4: CLASSIFY PARAMETERS INTO CLINICAL CATEGORIES")
print("=" * 80)

# Define specific categories with keywords and patterns
category_mapping = {
    # Index / Demographics
    'Index': {
        'keywords': ['wiek', 'group', 'subject_id', 'nr kg', 'rok kg', 'dokument', 'przyjęcie', 'wypis', 'oddział'],
        'patterns': []
    },
    
    # Androgen / Hormonal
    'Testosterone_Total': {
        'keywords': ['testosteron', 'testosterone'],
        'patterns': [r'testosteron', r'testosterone', r'l_testos']
    },
    'Testosterone_Free': {
        'keywords': ['wolny testosteron', 'free testosterone', 'test-f', 'o41.w'],
        'patterns': [r'wolny testosteron', r'free testosterone', r'test-f', r'o41\.w']
    },
    'SHBG': {
        'keywords': ['shbg', 'sex hormone binding globulin'],
        'patterns': [r'shbg', r'l_shgb']
    },
    'DHEAS': {
        'keywords': ['dheas', 'dhea', 'dehydroepiandrosterone'],
        'patterns': [r'dheas', r'dhea']
    },
    'Androstenedione': {
        'keywords': ['androstendion', 'androstenedione', 'andro'],
        'patterns': [r'androstendion', r'androstenedione', r'andro']
    },
    'AMH': {
        'keywords': ['amh', 'anty-mullerian', 'anti-mullerian'],
        'patterns': [r'amh', r'anty-mullerian', r'anti-mullerian']
    },
    'LH': {
        'keywords': ['lh', 'luteinizing hormone'],
        'patterns': [r'lh', r'l_lh']
    },
    'FSH': {
        'keywords': ['fsh', 'follicle stimulating'],
        'patterns': [r'fsh', r'l_fsh']
    },
    'Estradiol': {
        'keywords': ['estradiol', 'e2', 'estra'],
        'patterns': [r'estradiol', r'estra', r'e2']
    },
    'Progesterone_17OHP': {
        'keywords': ['17-oh progesteron', '17 ohp', 'l79', 'synacthen', '17-ohp'],
        'patterns': [r'17-oh', r'17ohp', r'l79', r'synacthen']
    },
    'Prolactin': {
        'keywords': ['prl', 'prolactin', 'makroprl'],
        'patterns': [r'prl', r'prolactin', r'makroprl']
    },
    
    # Metabolic / Glucose
    'Glucose_Fasting': {
        'keywords': ['glu 0', 'glucose 0', 'glukoza', 'l_glu_0', 'glu 0\''],
        'patterns': [r'glu 0', r'glucose 0', r'l_glu_0', r'glu 0\'', r'glukoza']
    },
    'Glucose_OGTT': {
        'keywords': ['dobowy rytm tolerancji glukozy', 'l_g', 'krzywa cukrowa'],
        'patterns': [r'\bl_g\d+\b', r'krzywa cukrowa', r'ogtt']
    },
    'Insulin_Fasting': {
        'keywords': ['insulina', 'insulin', 'insul', 'insulina uu/ml'],
        'patterns': [r'insulina', r'insulin', r'insul', r'insulina uu']
    },
    'Insulin_OGTT': {
        'keywords': ['insulina po', 'l97', 'insulin po 75g'],
        'patterns': [r'l97', r'insulina po', r'insulin po 75g']
    },
    'HOMA_IR': {
        'keywords': ['homa', 'homa-ir'],
        'patterns': [r'homa']
    },
    'QUICKI': {
        'keywords': ['quicki'],
        'patterns': [r'quicki']
    },
    'HbA1c': {
        'keywords': ['hemoglobina glikowana', 'hba1c', 'l53.ifc', 'glycated hemoglobin'],
        'patterns': [r'hba1c', r'hemoglobina glikowana', r'l53', r'glycated']
    },
    'TyG': {
        'keywords': ['tyg', 'triglyceride glucose'],
        'patterns': [r'tyg']
    },
    
    # Lipid
    'Cholesterol_Total': {
        'keywords': ['cholesterol całkowity', 'total cholesterol', 'tchol'],
        'patterns': [r'cholesterol całkowity', r'total cholesterol', r'tchol']
    },
    'HDL': {
        'keywords': ['hdl', 'hdl cholesterol'],
        'patterns': [r'hdl']
    },
    'LDL': {
        'keywords': ['ldl', 'ldl cholesterol'],
        'patterns': [r'ldl']
    },
    'Triglycerides': {
        'keywords': ['triglicerydy', 'triglycerides', 'tg'],
        'patterns': [r'triglicerydy', r'triglycerides', r'\btg\b']
    },
    
    # Inflammatory / Hematology
    'CRP': {
        'keywords': ['crp', 'białko c-reaktywne', 'c-reactive protein'],
        'patterns': [r'crp', r'białko c-reaktywne']
    },
    'NLR': {
        'keywords': ['nlr', 'neutrophil lymphocyte ratio'],
        'patterns': [r'nlr']
    },
    'PLR': {
        'keywords': ['plr', 'platelet lymphocyte ratio'],
        'patterns': [r'plr']
    },
    'Neutrophils': {
        'keywords': ['neut', 'neutrofile', 'neutrophil'],
        'patterns': [r'neu', r'neut', r'neutrofil']
    },
    'Lymphocytes': {
        'keywords': ['limf', 'lymph', 'lymphocyte'],
        'patterns': [r'limf', r'lymph']
    },
    'Monocytes': {
        'keywords': ['mono', 'monocyte'],
        'patterns': [r'mon', r'monocyte']
    },
    'Eosinophils': {
        'keywords': ['eos', 'eosinophil'],
        'patterns': [r'eos']
    },
    'Basophils': {
        'keywords': ['baso', 'basophil'],
        'patterns': [r'baso']
    },
    'Platelets': {
        'keywords': ['plt', 'płytki', 'thrombocyte', 'plt 10^3'],
        'patterns': [r'plt', r'płytki']
    },
    'WBC': {
        'keywords': ['wbc', 'leukocyty', 'white blood cell'],
        'patterns': [r'wbc', r'leukocyty']
    },
    'RBC': {
        'keywords': ['rbc', 'erythrocyte', 'krwinki czerwone'],
        'patterns': [r'rbc']
    },
    'Hemoglobin': {
        'keywords': ['hgb', 'hemoglobina'],
        'patterns': [r'hgb', r'hemoglobina']
    },
    'Hematocrit': {
        'keywords': ['hct', 'hematokryt'],
        'patterns': [r'hct', r'hematokryt']
    },
    'MCV': {
        'keywords': ['mcv', 'mean corpuscular volume'],
        'patterns': [r'mcv']
    },
    'MCH': {
        'keywords': ['mch', 'mean corpuscular hemoglobin'],
        'patterns': [r'mch']
    },
    'MCHC': {
        'keywords': ['mchc', 'mean corpuscular hemoglobin concentration'],
        'patterns': [r'mchc']
    },
    'RDW': {
        'keywords': ['rdw', 'red cell distribution width'],
        'patterns': [r'rdw']
    },
    'MPV': {
        'keywords': ['mpv', 'mean platelet volume', 'pct', 'pdw', 'plcr'],
        'patterns': [r'\bmpv\b', r'\bpct\b', r'\bpdw\b', r'\bplcr\b']
    },
    'Ferritin': {
        'keywords': ['ferrytyna', 'ferritin', 'ferr', 'l05'],
        'patterns': [r'ferrytyna', r'ferritin', r'ferr', r'l05']
    },
    'Iron': {
        'keywords': ['żelazo', 'iron', 'fe'],
        'patterns': [r'żelazo', r'iron', r'fe\b']
    },
    
    # Coagulation
    'D_Dimer': {
        'keywords': ['d-dimer', 'ddimer'],
        'patterns': [r'd-dimer', r'ddimer']
    },
    'Fibrinogen': {
        'keywords': ['fibrynogen', 'fibrinogen', 'l_fib'],
        'patterns': [r'fibrynogen', r'fibrinogen', r'l_fib']
    },
    'PT_APTT': {
        'keywords': ['protrombina', 'pt', 'aptt', 'inr'],
        'patterns': [r'protrombina', r'\bpt\b', r'aptt', r'inr']
    },
    
    # Liver
    'ALT': {
        'keywords': ['alt', 'alat'],
        'patterns': [r'alt', r'alat']
    },
    'AST': {
        'keywords': ['ast', 'aspat'],
        'patterns': [r'ast', r'aspat']
    },
    'GGTP': {
        'keywords': ['ggtp', 'gamma gt'],
        'patterns': [r'ggtp', r'gamma']
    },
    'Bilirubin': {
        'keywords': ['bilirubina', 'bilirubin'],
        'patterns': [r'bilirubina', r'bilirubin']
    },
    
    # Kidney
    'Creatinine': {
        'keywords': ['kreatynina', 'creatinine', 'kreat'],
        'patterns': [r'kreatynina', r'creatinine', r'kreat']
    },
    'GFR_MDRD': {
        'keywords': ['mdrd', 'gfr'],
        'patterns': [r'mdrd', r'gfr']
    },
    'Electrolytes': {
        'keywords': ['sód', 'potas', 'chlorki', 'jonogram'],
        'patterns': [r'sód', r'potas', r'chlorki', r'jonogram', r'\bl_na\b', r'\bl_k\b', r'\bl_cl\b']
    },
    
    # Thyroid
    'TSH': {
        'keywords': ['tsh', 'thyrotropin'],
        'patterns': [r'tsh']
    },
    'FT3': {
        'keywords': ['ft3', 'free t3'],
        'patterns': [r'ft3']
    },
    'FT4': {
        'keywords': ['ft4', 'free t4'],
        'patterns': [r'ft4']
    },
    'Anti_TPO': {
        'keywords': ['anty-tpo', 'anti-tpo', 'ata_tpo'],
        'patterns': [r'anty-tpo', r'anti-tpo', r'tpo']
    },
    'Anti_TG': {
        'keywords': ['anty-tg', 'anti-tg', 'antytyreoglobulinowe'],
        'patterns': [r'anty-tg', r'anti-tg', r'antytyreoglobulinowe', r'atg']
    },
    'TRAb': {
        'keywords': ['trab', 'tsh receptor antibody', 'przeciw receptorowi tsh'],
        'patterns': [r'trab', r'tsh receptor']
    },
    
    # Adrenal / Cortisol
    'Cortisol_8AM': {
        'keywords': ['kortyzol 8', 'kortyzol godz 08', 'kor 8', 'kortyzol 8.00', 'korcz'],
        'patterns': [r'kortyzol.*8', r'kor.*8', r'kor08', r'korcz']
    },
    'Cortisol_11PM': {
        'keywords': ['kortyzol 23', 'kortyzol godz 23', 'kor23'],
        'patterns': [r'kortyzol.*23', r'kor23']
    },
    'Cortisol_Dexamethasone': {
        'keywords': ['kord', 'dexamethasone', 'deksametazon', 'kortyzol po dex'],
        'patterns': [r'kord', r'dexamethasone', r'deksametazon', r'kortyzol po']
    },
    'ACTH_Stimulation': {
        'keywords': ['synacthen', 'l79', 'test z synacthenem'],
        'patterns': [r'synacthen', r'l79', r'test z synacthenem']
    },
    
    # Growth Factors
    'IGF1': {
        'keywords': ['igf-1', 'igf1', 'insulin like growth factor'],
        'patterns': [r'igf-?1', r'insulin like growth']
    },
    'GH': {
        'keywords': ['gh', 'growth hormone', 'hormon wzrostu'],
        'patterns': [r'gh\b', r'growth hormone', r'hormon wzrostu']
    },
    
    # Tumor Markers
    'CA125': {
        'keywords': ['ca125', 'ca 125'],
        'patterns': [r'ca125', r'ca 125']
    },
    'CA19_9': {
        'keywords': ['ca19.9', 'ca 19.9'],
        'patterns': [r'ca19\.9', r'ca 19\.9']
    },
    'CEA': {
        'keywords': ['cea'],
        'patterns': [r'cea']
    },
    'HE4_ROMA': {
        'keywords': ['he4', 'roma', 'test roma'],
        'patterns': [r'he4', r'roma']
    },
    
    # Immunology / Autoimmunity
    'Anti_HCV': {
        'keywords': ['anty-hcv', 'anti-hcv', 'l_anthcv'],
        'patterns': [r'anty-hcv', r'anti-hcv', r'anthcv']
    },
    'HBsAg': {
        'keywords': ['hbsag', 'hbs ag', 'hepatitis b'],
        'patterns': [r'hbsag', r'hbs']
    },
    'Anti_GAD': {
        'keywords': ['pcgad', 'anti-gad', 'przeciw gad'],
        'patterns': [r'pcgad', r'anti-gad', r'przeciw gad']
    },
    'ASO': {
        'keywords': ['aso', 'antistreptolysin'],
        'patterns': [r'aso']
    },
    
    # Urinalysis
    'Urinalysis': {
        'keywords': ['mocz', 'urine', 'mbarwa', 'mbialk', 'mcieza', 'mcukie', 'mery', 'mketon', 'mleu', 'mnit', 'mosad', 'mph', 'mprzej', 'murobi'],
        'patterns': [r'mb\w+|mocz|urine', r'mcieza|mcukie|mosad|mph|mleu|mnit|mery|mketon|mbarwa|mbialk|mprzej|murobi']
    },
    
    # Imaging
    'Ultrasound': {
        'keywords': ['usg', 'ultrasound', 'elastografia'],
        'patterns': [r'usg', r'ultrasound', r'elastografia']
    },
    'CT_Scan': {
        'keywords': ['tk', 'ct scan', 'tomografia'],
        'patterns': [r'tk jamy', r'ct scan', r'tomografia']
    },
    'XRay': {
        'keywords': ['rtg', 'x-ray', 'radiograph'],
        'patterns': [r'rtg', r'x-ray']
    },
    
    # Microbiology
    'Microbiology': {
        'keywords': ['posiew', 'wymaz', 'culture', 'swab', 'dat-zak', 'uwagi'],
        'patterns': [r'posiew', r'wymaz', r'culture', r'swab', r'dat-zak', r'uwagi']
    },
    
    # Vitamins / Other
    'Vitamin_D': {
        'keywords': ['witamina d', 'vitamin d', '25-hydroksywitamina', 'witd'],
        'patterns': [r'witamina d', r'vitamin d', r'25-hydroksy', r'witd']
    },
    'Calcium': {
        'keywords': ['wapń', 'calcium', 'tcal', 'l_cazjs'],
        'patterns': [r'wapń', r'calcium', r'tcal', r'cazjs']
    },
    'Phosphate': {
        'keywords': ['fosforany', 'phosphate', 'fosfor'],
        'patterns': [r'fosforany', r'phosphate', r'fosfor']
    },
    'Beta_HCG': {
        'keywords': ['beta hcg', 'bhcg', 'l_hcg'],
        'patterns': [r'beta hcg', r'bhcg', r'l_hcg']
    },
    
    # COVID / Respiratory
    'COVID_Respiratory': {
        'keywords': ['covid', 'sars-cov', 'grypa', 'rsv', 'cov2'],
        'patterns': [r'covid', r'sars-cov', r'grypa', r'rsv', r'cov2']
    },
    
    'Immature_Granulocytes': {
        'keywords': ['ig_b', 'ig_p'],
        'patterns': [r'\big_b\b', r'\big_p\b']
    },

    'Lymphocytes': {
        'keywords': ['limf', 'lim#', 'lim ', 'lymph'],
        'patterns': [r'\blim#\b', r'\blim\b', r'limf', r'lymph']
    },
}

# Initialize category assignment
for col, info in parameter_quality.items():
    if col in ['group', 'subject_id', 'nr', 'Wiek', 'BMI kg/m2']:
        info['category'] = 'Index'
        continue
    
    info['category'] = None
    col_lower = col.lower()
    
    for category, mapping in category_mapping.items():
        # Check keywords
        matched = False
        for keyword in mapping['keywords']:
            if keyword.lower() in col_lower:
                info['category'] = category
                matched = True
                break
        
        # Check regex patterns if no keyword match
        if not matched and mapping['patterns']:
            for pattern in mapping['patterns']:
                if re.search(pattern, col_lower, re.IGNORECASE):
                    info['category'] = category
                    matched = True
                    break
        
        if matched:
            break

# Collect uncategorized columns
uncategorized = []
for col, info in parameter_quality.items():
    if info['category'] is None and col not in ['group', 'subject_id', 'nr', 'Wiek', 'BMI kg/m2']:
        uncategorized.append(col)
        info['category'] = 'UNCATEGORIZED'
        info['notes'].append('REVIEW MANUALLY - no category matched')

# Display results
print("\n" + "=" * 80)
print("CATEGORIZATION RESULTS")
print("=" * 80)

category_counts = {}
for info in parameter_quality.values():
    cat = info['category']
    if cat:
        category_counts[cat] = category_counts.get(cat, 0) + 1

print("\nCategory counts (including UNCATEGORIZED):")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

# Display all columns with their categories
print("\n" + "=" * 80)
print("FULL CATEGORIZATION LIST")
print("=" * 80)

for i, (col, info) in enumerate(list(parameter_quality.items())):
    print(f"{i+1:3}. {col} -> {info['category']}")

# Display ONLY uncategorized columns (MUST REVIEW)
print("\n" + "=" * 80)
print(f"⚠️ UNCATEGORIZED COLUMNS TO REVIEW ({len(uncategorized)})")
print("=" * 80)

if uncategorized:
    for col in uncategorized:
        info = parameter_quality[col]
        print(f"  ❌ {col}")
        print(f"     N={info['n_nonnull']}, type={info['data_type']}")
        print(f"     Sample: {df[col].dropna().head(3).tolist()}")
        print()
else:
    print("✅ No uncategorized columns found!")

# Update parameter_quality with category info
print(f"\n✅ Updated parameter_quality with categories for {len([i for i in parameter_quality.values() if i['category']])} columns")
print(f"⚠️ {len(uncategorized)} columns need manual review")

STEP 4: CLASSIFY PARAMETERS INTO CLINICAL CATEGORIES

CATEGORIZATION RESULTS

Category counts (including UNCATEGORIZED):
  Insulin_Fasting: 13
  Urinalysis: 13
  Glucose_OGTT: 10
  Index: 9
  MPV: 9
  Prolactin: 8
  PT_APTT: 7
  RBC: 6
  HE4_ROMA: 6
  Microbiology: 6
  Progesterone_17OHP: 5
  Triglycerides: 5
  FSH: 5
  LH: 5
  RDW: 5
  AST: 4
  TyG: 4
  HbA1c: 4
  Cortisol_Dexamethasone: 4
  MCH: 4
  Testosterone_Total: 4
  Ultrasound: 4
  AMH: 3
  Androstenedione: 3
  Glucose_Fasting: 3
  HBsAg: 3
  Electrolytes: 3
  Cortisol_8AM: 3
  Lymphocytes: 3
  Platelets: 3
  WBC: 3
  Neutrophils: 3
  SHBG: 3
  CA125: 2
  DHEAS: 2
  Estradiol: 2
  Ferritin: 2
  Hematocrit: 2
  Hemoglobin: 2
  MCV: 2
  Basophils: 2
  Eosinophils: 2
  Immature_Granulocytes: 2
  Monocytes: 2
  TSH: 2
  Calcium: 2
  ALT: 1
  ASO: 1
  Anti_HCV: 1
  Anti_TPO: 1
  CRP: 1
  Bilirubin: 1
  CEA: 1
  CA19_9: 1
  Cholesterol_Total: 1
  D_Dimer: 1
  FT3: 1
  FT4: 1
  Fibrinogen: 1
  Phosphate: 1
  HDL: 1
  Cortisol_11PM: 1

In [52]:
# STEP 5: PARAMETERS WITH VERY FEW DATA (Two Thresholds)
print("=" * 80)
print("STEP 5: PARAMETERS WITH VERY FEW DATA (Two Thresholds)")
print("=" * 80)

# Define thresholds
EXCLUSION_THRESHOLD = 10  # N < 10 -> exclude automatically
WARNING_THRESHOLD = 30    # 10 ≤ N < 30 -> manual verification needed

# Collect parameters by threshold
exclude_params = []      # N < 10
verify_params = []       # 10 ≤ N < 30
sufficient_params = []   # N ≥ 30

for col, info in parameter_quality.items():
    if col in ['group', 'subject_id', 'nr', 'Wiek', 'BMI kg/m2']:
        continue
    
    n = info['n_nonnull']
    
    if n == 0:
        exclude_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'N={n} - completely empty column (no data)')
        info['recommended_action'] = 'exclude'
        info['notes'].append(f'No non-null values at all')

    elif n < EXCLUSION_THRESHOLD:
        exclude_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'N={n} < {EXCLUSION_THRESHOLD} - automatic exclusion')
        info['recommended_action'] = 'exclude'
        info['notes'].append(f'Only {n} non-null values')
    elif n < WARNING_THRESHOLD:
        verify_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False  # Not useful for main analysis, but keep for manual review
        info['exclude_reason'].append(f'N={n} between {EXCLUSION_THRESHOLD} and {WARNING_THRESHOLD} - needs manual verification')
        info['recommended_action'] = 'manual_verify'
        info['notes'].append(f'Only {n} non-null values - consider if useful for subgroup analysis')
    else:
        sufficient_params.append(col)

# Display results
print(f"\nThresholds:")
print(f"  ❌ EXCLUDE (N < {EXCLUSION_THRESHOLD}): {len(exclude_params)} parameters")
print(f"  ⚠️ VERIFY ({EXCLUSION_THRESHOLD} ≤ N < {WARNING_THRESHOLD}): {len(verify_params)} parameters")
print(f"  ✅ SUFFICIENT (N ≥ {WARNING_THRESHOLD}): {len(sufficient_params)} parameters")

# ============================================================================
# DETAILED LIST - COMPLETELY EMPTY COLUMNS (N = 0)
# ============================================================================
print("\n" + "=" * 80)
print("📭 COMPLETELY EMPTY COLUMNS (N = 0)")
print("=" * 80)

empty_cols = [col for col in exclude_params if parameter_quality[col]['n_nonnull'] == 0]

if empty_cols:
    for col in empty_cols:
        info = parameter_quality[col]
        print(f"  {col}")
        print(f"     Category: {info['category']}")
        print(f"     Reason: {info['exclude_reason'][-1]}")
else:
    print("  No completely empty columns found")

# ============================================================================
# DETAILED LIST - EXCLUDE (N < 10)
# ============================================================================
print("\n" + "=" * 80)
print(f"❌ EXCLUDE (N < {EXCLUSION_THRESHOLD}) - AUTOMATIC EXCLUSION")
print("=" * 80)

if exclude_params:
    for col in sorted(exclude_params, key=lambda x: parameter_quality[x]['n_nonnull']):
        info = parameter_quality[col]
        print(f"\n  {col}")
        print(f"     Category: {info['category']}")
        print(f"     N={info['n_nonnull']}, unique={info['n_unique']}")
        print(f"     Sample values: {df[col].dropna().head(3).tolist()}")
else:
    print("  No parameters in this category")

# ============================================================================
# DETAILED LIST - MANUAL VERIFY (10 ≤ N < 30)
# ============================================================================
print("\n" + "=" * 80)
print(f"⚠️  MANUAL VERIFY ({EXCLUSION_THRESHOLD} ≤ N < {WARNING_THRESHOLD})")
print("=" * 80)

if verify_params:
    # Group by category
    verify_by_category = {}
    for col in verify_params:
        cat = parameter_quality[col]['category']
        if cat not in verify_by_category:
            verify_by_category[cat] = []
        verify_by_category[cat].append((col, parameter_quality[col]['n_nonnull']))
    
    for cat, cols in sorted(verify_by_category.items(), key=lambda x: -len(x[1])):
        print(f"\n  {cat} ({len(cols)} parameters):")
        for col, n in sorted(cols, key=lambda x: x[1]):
            info = parameter_quality[col]
            print(f"     {col}: N={n}, unique={info['n_unique']}")
else:
    print("  No parameters in this category")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

print(f"\n{'Category':<25} {'Exclude':<10} {'Verify':<10} {'Sufficient':<10} {'Total':<10}")
print("-" * 65)

# Get categories from all parameters
all_categories = set()
for info in parameter_quality.values():
    if info['category'] and info['category'] not in ['Index', 'UNCATEGORIZED']:
        all_categories.add(info['category'])

category_summary = {}
for cat in sorted(all_categories):
    exclude_count = sum(1 for col in exclude_params if parameter_quality[col].get('category') == cat)
    verify_count = sum(1 for col in verify_params if parameter_quality[col].get('category') == cat)
    sufficient_count = sum(1 for col in sufficient_params if parameter_quality[col].get('category') == cat and parameter_quality[col].get('category') == cat)
    total = exclude_count + verify_count + sufficient_count
    if total > 0:
        print(f"{cat:<25} {exclude_count:<10} {verify_count:<10} {sufficient_count:<10} {total:<10}")


STEP 5: PARAMETERS WITH VERY FEW DATA (Two Thresholds)

Thresholds:
  ❌ EXCLUDE (N < 10): 78 parameters
  ⚠️ VERIFY (10 ≤ N < 30): 17 parameters
  ✅ SUFFICIENT (N ≥ 30): 125 parameters

📭 COMPLETELY EMPTY COLUMNS (N = 0)
  Mocz - badanie ogólne (MOSAD)
     Category: Urinalysis
     Reason: N=0 - completely empty column (no data)
  NLR
     Category: NLR
     Reason: N=0 - completely empty column (no data)
  PLR
     Category: PLR
     Reason: N=0 - completely empty column (no data)
  RTG kości nadgarstka, PA, bok (nan)
     Category: Triglycerides
     Reason: N=0 - completely empty column (no data)
  TK jamy brzusznej i miednicy z kontrastem (nan)
     Category: AST
     Reason: N=0 - completely empty column (no data)
  TK jamy brzusznej z kontrastem (nan)
     Category: AST
     Reason: N=0 - completely empty column (no data)
  USG brzucha i przestrzeni zaotrzewnowej (nan)
     Category: Ultrasound
     Reason: N=0 - completely empty column (no data)
  USG nadnerczy (nan)
     Categ

## Manual Verification Decision Summary (Step 5)

**Decision:** Exclude all 17 parameters from the `verify` category.

Based on the review, all parameters with **N between 10 and 30** were excluded from further analysis. The rationale is documented below.

---

## Rationale by Category

| Category | Parameters (N) | Decision | Justification |
|---|---|---|---|
| Prolactin | MAKROPRL (14), PRL16 (23), PRL10 (25), PRL10PEG (25) | ❌ Exclude | Better alternatives exist: `PRL godz. 10:00 (PRL10)` for PCOS group, `PRL ng/ml` for Control group |
| FSH (OGTT) | L_FSH0, L_FSH30, L_FSH60 (15 each) | ❌ Exclude | Small N; FSH information available from `FSH (FSH)` (PCOS) and `FSH IU/l` (Control) |
| LH (OGTT) | LH0 (15), L_LH30 (15), L_LH60 (15) | ❌ Exclude | Small N; LH information available from `LH (LH)` (PCOS) and `LH IU/l` (Control) |
| HbA1c | HBA1C_1 (23), HBA1C_2 (23) | ❌ Exclude | Insufficient sample size for meaningful analysis |
| Triglycerides | Anty-TG (O18) (21) | ❌ Exclude | Other TG columns have more data |
| CRP | Białko C-reaktywne (CRP) (19) | ❌ Exclude | Insufficient sample size |
| Index | Dokument (NAZWA) (18) | ❌ Exclude | Index/metadata column, not analytical; small N anyway |
| HE4_ROMA | Test Roma (HE4) (10) | ❌ Exclude | Very small N (only 10) – insufficient for analysis |
| Beta_HCG | beta HCG (L_HCG) (21) | ❌ Exclude | Only column for this parameter but highly incomplete; insufficient for meaningful analysis |

---

## Final Count

| Status | Count |
|---|---|
| ✅ Keep (Sufficient N ≥ 30) | 125 |
| ❌ Exclude (N < 10) | 78 |
| ❌ Exclude (Manual verify, 10–30) | 17 |
| **Total columns processed** | **220** |

In [53]:
print("=" * 80)
print("FLAGGING MANUAL VERIFY PARAMETERS AS EXCLUDED")
print("=" * 80)

# List of parameters to exclude from manual verify category
manual_exclude_list = [
    # Prolactin (4)
    'Procent odzysku prolaktyny po precypitacji (MAKROPRL)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL16)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL10)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL10PEG)',
    
    # FSH (3)
    'FSH 0\' 30\' 60\' (L_FSH0)',
    'FSH 0\' 30\' 60\' (L_FSH30)',
    'FSH 0\' 30\' 60\' (L_FSH60)',
    
    # LH (3)
    'LH 0\' 30\' 60\' (LH0)',
    'LH 0\' 30\' 60\' (L_LH30)',
    'LH 0\' 30\' 60\' (L_LH60)',
    
    # HbA1c (2)
    'Hemoglobina glikowana (HBA1C_1)',
    'Hemoglobina glikowana (HBA1C_2)',
    
    # Triglycerides / Anti-TG (1)
    'Anty-TG (O18)',
    
    # CRP (1)
    'Białko C-reaktywne (CRP)',
    
    # Index (1)
    'Dokument (NAZWA)',
    
    # HE4_ROMA (1)
    'Test Roma (HE4)',
    
    # Beta_HCG (1)
    'beta HCG (L_HCG)'
]

# Update parameter_quality for each
for col in manual_exclude_list:
    if col in parameter_quality:
        info = parameter_quality[col]
        info['useful'] = False
        info['too_few_data'] = True
        info['exclude_reason'].append(f'Manual decision: excluded after review (N={info["n_nonnull"]} between 10-30)')
        info['recommended_action'] = 'exclude'
        info['notes'].append('Manual exclude decision - insufficient data or better alternatives exist')
        print(f"  ❌ {col} (N={info['n_nonnull']}) -> excluded")

# Verify they are now marked as excluded
print("\n" + "=" * 80)
print("VERIFICATION: Parameters marked as 'useful = False'")
print("=" * 80)

verify_count = sum(1 for col in manual_exclude_list if col in parameter_quality and not parameter_quality[col]['useful'])
print(f"\n✅ Successfully flagged {verify_count} / {len(manual_exclude_list)} parameters as excluded")

# Final count of useful parameters
final_useful = sum(1 for info in parameter_quality.values() if info.get('useful') == True and info.get('category') not in ['Index'])
print(f"\n📊 FINAL COUNT - Useful parameters for analysis: {final_useful}")

FLAGGING MANUAL VERIFY PARAMETERS AS EXCLUDED
  ❌ Procent odzysku prolaktyny po precypitacji (MAKROPRL) (N=14) -> excluded
  ❌ PRL 10:00, 10:00 po PEG, 16:00 (PRL16) (N=23) -> excluded
  ❌ PRL 10:00, 10:00 po PEG, 16:00 (PRL10) (N=25) -> excluded
  ❌ PRL 10:00, 10:00 po PEG, 16:00 (PRL10PEG) (N=25) -> excluded
  ❌ FSH 0' 30' 60' (L_FSH0) (N=15) -> excluded
  ❌ FSH 0' 30' 60' (L_FSH30) (N=15) -> excluded
  ❌ FSH 0' 30' 60' (L_FSH60) (N=15) -> excluded
  ❌ LH 0' 30' 60' (LH0) (N=15) -> excluded
  ❌ LH 0' 30' 60' (L_LH30) (N=15) -> excluded
  ❌ LH 0' 30' 60' (L_LH60) (N=15) -> excluded
  ❌ Hemoglobina glikowana (HBA1C_1) (N=23) -> excluded
  ❌ Hemoglobina glikowana (HBA1C_2) (N=23) -> excluded
  ❌ Anty-TG (O18) (N=21) -> excluded
  ❌ Białko C-reaktywne (CRP) (N=19) -> excluded
  ❌ Dokument (NAZWA) (N=18) -> excluded
  ❌ Test Roma (HE4) (N=10) -> excluded
  ❌ beta HCG (L_HCG) (N=21) -> excluded

VERIFICATION: Parameters marked as 'useful = False'

✅ Successfully flagged 17 / 17 parameters 

In [ ]:
# STEP 6: DETECT FORMATTING ERRORS (optimized for your data types)

print("=" * 80)
print("STEP 6: DETECT FORMATTING ERRORS")
print("=" * 80)

# Get list of useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

print(f"Checking {len(useful_params)} useful parameters for formatting issues...")

# Text null patterns to check for
null_patterns = ['null', 'NULL', 'NaN', 'nan', 'None', 'N/A', 'n/a', '?', '--', '\\N', 'NA', 'Na', 'missing', 'Missing']

formatting_issues = []

for col in useful_params:
    info = parameter_quality[col]
    errors_found = []
    sample = df[col].dropna()
    
    if len(sample) == 0:
        continue
    
    # ===== CHECK 1: Numeric columns (float64, int64) =====
    if df[col].dtype in ['float64', 'int64']:
        # Check for negative values
        negative_count = (sample < 0).sum()
        if negative_count > 0:
            errors_found.append(f'contains negative values - {negative_count} occurrences')
        
    
    # ===== CHECK 2: String columns (str) =====
    elif df[col].dtype == 'str' or df[col].dtype == 'object':
        sample_str = sample.astype(str)
        
        # Check for text null values
        text_null_count = sample_str.isin(null_patterns).sum()
        if text_null_count > 0:
            null_example = sample_str[sample_str.isin(null_patterns)].iloc[0]
            errors_found.append(f'contains text null values (e.g., "{null_example}") - {text_null_count} occurrences')
        
        # Check for commas (Polish decimal format)
        comma_count = sample_str.str.contains(',').sum()
        if comma_count > 0:
            errors_found.append(f'contains commas (Polish decimal) - {comma_count} occurrences')
        
        # Check for special characters
        special_chars = sample_str.str.contains(r'[><~]', na=False).sum()
        if special_chars > 0:
            errors_found.append(f'contains special characters (>, <, ~) - {special_chars} occurrences')
        
        # Check if column could be numeric (most values are numbers)
        numeric_count = 0
        text_count = 0
        for val in sample_str.head(50):
            try:
                float(str(val).replace(',', '.'))
                numeric_count += 1
            except:
                text_count += 1
        
        if numeric_count > 0 and text_count > 0:
            errors_found.append(f'mixed types: {numeric_count} numeric, {text_count} text in sample (consider converting)')
        
        # Check for placeholder values
        placeholder_count = sample_str.str.contains(r'^\?$|^\.\.\.$|^xxx$|^---$', na=False, regex=True).sum()
        if placeholder_count > 0:
            errors_found.append(f'contains placeholder values - {placeholder_count} occurrences')
    
    
    if errors_found:
        formatting_issues.append({
            'column': col,
            'category': info['category'],
            'dtype': str(df[col].dtype),
            'errors': errors_found,
            'sample': sample.head(5).tolist(),
            'n_nonnull': info['n_nonnull']
        })
        info['formatting_errors'] = True
        info['exclude_reason'].extend(errors_found)
        info['recommended_action'] = 'fix_formatting'
        info['notes'].append(f'Formatting errors: {", ".join(errors_found)}')
        print(f"\n⚠️ {col} [{info['category']}] (dtype={df[col].dtype})")
        for err in errors_found:
            print(f"     - {err}")

# Display summary
print("\n" + "=" * 80)
print("FORMATTING ISSUES SUMMARY")
print("=" * 80)

print(f"\nFound {len(formatting_issues)} columns with formatting issues out of {len(useful_params)} useful parameters")

if formatting_issues:
    print("\nColumns with formatting issues:")
    for issue in formatting_issues:
        print(f"  • {issue['column']} ({issue['category']}) - dtype={issue['dtype']}, N={issue['n_nonnull']}")
        for err in issue['errors']:
            print(f"      - {err}")



STEP 6: DETECT FORMATTING ERRORS
Checking 121 useful parameters for formatting issues...

⚠️ Morfologia CBC (NRBC_B) [RBC] (dtype=str)
     - contains commas (Polish decimal) - 44 occurrences

⚠️ Morfologia CBC (NRBC_P) [RBC] (dtype=str)
     - contains commas (Polish decimal) - 44 occurrences

⚠️ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B) [RBC] (dtype=str)
     - contains commas (Polish decimal) - 901 occurrences

⚠️ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_P) [RBC] (dtype=str)
     - contains commas (Polish decimal) - 902 occurrences

⚠️ Witamina D (25- hydroksywitamina D3) (WITD) [Vitamin_D] (dtype=str)
     - contains commas (Polish decimal) - 809 occurrences
     - contains special characters (>, <, ~) - 1 occurrences

FORMATTING ISSUES SUMMARY

Found 5 columns with formatting issues out of 121 useful parameters

Columns with formatting issues:
  • Morfologia CBC (NRBC_B) (RBC) -

## Step 6: Formatting Errors – Decisions Needed

The following columns contain formatting issues that prevent automatic conversion to numeric type. **A decision is required** on how to handle the problematic values.

### Columns Requiring Data Cleaning Decisions

| Column | Category | N | Issue | Problematic Values |
|--------|----------|---|-------|-------------------|
| `Morfologia CBC (NRBC_B)` | RBC | 44 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Morfologia CBC (NRBC_P)` | RBC | 44 | Commas | Clean comma values only (no mixed formats observed) |
| `Morfologia krwi... (NRBC_B)` | RBC | 901 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Morfologia krwi... (NRBC_P)` | RBC | 902 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Witamina D (WITD)` | Vitamin_D | 809 | Commas + special character | <3,00 \| 4,48 (contains < and \|) |

### Decisions Required

1. **For NRBC columns with `"0,00 | 0,01"`:**
   - Which value should be kept? (first? second? both? average?)
   - Should these be treated as missing?

2. **For Vitamin D with `"<3,00 | 4,48"`:**
   - How to handle the `<` (below detection limit)? (impute as 3.00? treat as missing? keep as categorical?)
   - Which value to keep? (first? second? both?)


In [55]:
print("=" * 80)
print("STEP 6: DATA QUALITY ISSUES – DECISIONS NEEDED")
print("=" * 80)

# List of columns with formatting issues requiring decisions
columns_needing_decisions = {
    'Morfologia CBC (NRBC_B)': {
        'category': 'RBC',
        'n': 44,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00 | 0,01', '0,00']
    },
    'Morfologia CBC (NRBC_P)': {
        'category': 'RBC',
        'n': 44,
        'issue': 'contains commas (Polish decimal)',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B)': {
        'category': 'RBC',
        'n': 901,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_P)': {
        'category': 'RBC',
        'n': 902,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Witamina D (25- hydroksywitamina D3) (WITD)': {
        'category': 'Vitamin_D',
        'n': 809,
        'issue': 'contains commas (Polish decimal) + contains special character "<"',
        'sample': ['23,80', '13,00', '41,80', '17,90', '18,90']
    }
}

# Update parameter_quality with notes
for col, details in columns_needing_decisions.items():
    if col in parameter_quality:
        info = parameter_quality[col]
        info['formatting_errors'] = True
        info['recommended_action'] = 'decision_needed'
        info['notes'].append(f'Data cleaning decision needed: {details["issue"]}')
        info['notes'].append(f'Problematic sample values: {details["sample"]}')
        print(f"\n❓ {col}")
        print(f"     Category: {details['category']}")
        print(f"     N={details['n']}")
        print(f"     Issue: {details['issue']}")
        print(f"     Sample: {details['sample']}")
        print(f"     → Decision needed on how to handle problematic values before numeric conversion")

STEP 6: DATA QUALITY ISSUES – DECISIONS NEEDED

❓ Morfologia CBC (NRBC_B)
     Category: RBC
     N=44
     Issue: contains commas (Polish decimal) + problematic value "0,00 | 0,01"
     Sample: ['0,00', '0,00', '0,00', '0,00 | 0,01', '0,00']
     → Decision needed on how to handle problematic values before numeric conversion

❓ Morfologia CBC (NRBC_P)
     Category: RBC
     N=44
     Issue: contains commas (Polish decimal)
     Sample: ['0,00', '0,00', '0,00', '0,00', '0,00']
     → Decision needed on how to handle problematic values before numeric conversion

❓ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B)
     Category: RBC
     N=901
     Issue: contains commas (Polish decimal) + problematic value "0,00 | 0,01"
     Sample: ['0,00', '0,00', '0,00', '0,00', '0,00']
     → Decision needed on how to handle problematic values before numeric conversion

❓ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Di

In [58]:
# STEP 7: PARAMETERS WITH NO VARIANCE (≤2 UNIQUE VALUES)

print("=" * 80)
print("STEP 7: PARAMETERS WITH NO VARIANCE (≤2 UNIQUE VALUES)")
print("=" * 80)

# Get list of useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

print(f"Checking {len(useful_params)} useful parameters for low variance...")

no_variance_params = []
low_variance_params = []  # exactly 2 unique values

for col in useful_params:
    info = parameter_quality[col]
    n_unique = info['n_unique']
    n_nonnull = info['n_nonnull']
    
    if n_nonnull == 0:
        continue
    
    if n_unique == 1:
        no_variance_params.append({
            'column': col,
            'category': info['category'],
            'n_unique': n_unique,
            'n_nonnull': n_nonnull,
            'constant_value': df[col].dropna().iloc[0] if n_nonnull > 0 else None
        })
        info['no_variance'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'no variance - only 1 unique value (constant: {df[col].dropna().iloc[0]})')
        info['recommended_action'] = 'exclude'
        
    elif n_unique == 2:
        low_variance_params.append({
            'column': col,
            'category': info['category'],
            'n_unique': n_unique,
            'n_nonnull': n_nonnull,
            'unique_values': df[col].dropna().unique().tolist()
        })
        # Don't automatically exclude - these might be binary flags
        info['notes'].append(f'low variance - only 2 unique values: {df[col].dropna().unique().tolist()}')

# Display results
print(f"\n✅ Parameters with NO variance (1 unique value): {len(no_variance_params)}")
print(f"⚠️ Parameters with LOW variance (2 unique values): {len(low_variance_params)}")

# ============================================================================
# DETAILED LIST - NO VARIANCE (CONSTANT COLUMNS)
# ============================================================================
print("\n" + "=" * 80)
print("❌ NO VARIANCE (1 UNIQUE VALUE) - AUTOMATIC EXCLUSION")
print("=" * 80)

if no_variance_params:
    for param in no_variance_params:
        print(f"\n  {param['column']}")
        print(f"     Category: {param['category']}")
        print(f"     N={param['n_nonnull']}, unique values=1")
        print(f"     Constant value: {param['constant_value']}")
        print(f"     → Excluded from further analysis")
else:
    print("  No constant columns found")

# ============================================================================
# DETAILED LIST - LOW VARIANCE (2 UNIQUE VALUES) - FOR REVIEW
# ============================================================================
print("\n" + "=" * 80)
print("⚠️ LOW VARIANCE (2 UNIQUE VALUES) - MANUAL REVIEW")
print("=" * 80)

if low_variance_params:
    # Group by category
    low_var_by_category = {}
    for param in low_variance_params:
        cat = param['category']
        if cat not in low_var_by_category:
            low_var_by_category[cat] = []
        low_var_by_category[cat].append(param)
    
    for cat, params in sorted(low_var_by_category.items()):
        print(f"\n  {cat} ({len(params)} parameters):")
        for param in params:
            print(f"     • {param['column']}")
            print(f"       N={param['n_nonnull']}, unique values: {param['unique_values']}")
else:
    print("  No parameters with exactly 2 unique values found")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("STEP 7 SUMMARY")
print("=" * 80)

print(f"""
Parameters with NO variance (excluded automatically): {len(no_variance_params)}
Parameters with LOW variance (2 values) - review needed: {len(low_variance_params)}

Remaining useful parameters after Step 7: {len([col for col, info in parameter_quality.items() if info.get('useful') == True and info.get('category') not in ['Index']])}
""")


STEP 7: PARAMETERS WITH NO VARIANCE (≤2 UNIQUE VALUES)
Checking 117 useful parameters for low variance...

✅ Parameters with NO variance (1 unique value): 0
⚠️ Parameters with LOW variance (2 unique values): 1

❌ NO VARIANCE (1 UNIQUE VALUE) - AUTOMATIC EXCLUSION
  No constant columns found

⚠️ LOW VARIANCE (2 UNIQUE VALUES) - MANUAL REVIEW

  HBsAg (1 parameters):
     • HBsAg (L_HBSAG)
       N=1113, unique values: ['Niereaktywny', 'ujemny']

STEP 7 SUMMARY

Parameters with NO variance (excluded automatically): 0
Parameters with LOW variance (2 values) - review needed: 1

Remaining useful parameters after Step 7: 117



## Step 7: Parameters with No / Low Variance – Summary

### Results

| Category | Count | Action |
|----------|-------|--------|
| No variance (1 unique value) | 0 | – |
| Low variance (2 unique values) | 1 | Manual review |
| Normal variance (≥3 unique values) | 116 | Keep |

### Low Variance Parameter – Review Decision

| Column | Category | N | Unique Values | Decision |
|--------|----------|---|---------------|----------|
| `HBsAg (L_HBSAG)` | HBsAg | 1113 | `['Niereaktywny', 'ujemny']` | ✅ **Keep** – both values mean "negative" (clinically meaningful binary variable) |

### Final Status

- **Parameters remaining after Step 7:** 117
- **No automatic exclusions** in this step
- One parameter reviewed and confirmed as keep

In [59]:
# Update parameter_quality for this column
if 'HBsAg (L_HBSAG)' in parameter_quality:
    parameter_quality['HBsAg (L_HBSAG)']['notes'].append(
        'Manual review: two unique values ("Niereaktywny" and "ujemny") both mean "negative". No positive cases detected. '
        'Column can be kept as binary flag or consolidated to a single value.'
    )
    parameter_quality['HBsAg (L_HBSAG)']['recommended_action'] = 'keep_or_consolidate'